<a href="https://colab.research.google.com/github/SantiagoAmaya99/Crack-Detector-Using-Deep-Learning-Models/blob/main/Prueba_de_modelos_Detecci%C3%B3n_de_grietas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Dataset_IA2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Dataset_IA2


In [ ]:
#@title Librerias
import tensorflow as tf
import pandas as pd
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications import ConvNeXtTiny

In [ ]:
rows = []
for material in materiales:
    for clase in clases:
        folder_path = os.path.join(base_dir, material, clase)
        for img_path in glob.glob(os.path.join(folder_path, '*.*')):
            fname = os.path.basename(img_path)
            rows.append({
                "path": img_path,
                "label": 1 if clase == 'Cracked' else 0,
                "surface": material,
                "group": f"{material}_{fname.split('-')[0]}",
            })
df = pd.DataFrame(rows)

def group_split(frame, n_splits, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(frame, frame["label"], frame["group"]))
    return frame.iloc[a], frame.iloc[b]

trainval_df, test_df = group_split(df, 7)       # ~14% test
train_df, val_df = group_split(trainval_df, 6)  # ~14% val

# Verificación anti-fuga
assert set(train_df.group).isdisjoint(val_df.group)
assert set(train_df.group).isdisjoint(test_df.group)
assert set(val_df.group).isdisjoint(test_df.group)

for n, p in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, len(p), f"{p['label'].mean():.3f} con grieta")

train 40179 0.144 con grieta
val 8118 0.164 con grieta
test 7795 0.177 con grieta


In [ ]:
   for n, p in [("train", train_df), ("val", val_df), ("test", test_df)]:
       print(n, p['group'].nunique(), "fotos |", len(p), "imgs |",
             f"{p['label'].mean():.3f} con grieta")
       print(p['surface'].value_counts().to_dict())

train 165 fotos | 40179 imgs | 0.144 con grieta
{'Pavements': 18250, 'Walls': 11837, 'Decks': 10092}
val 33 fotos | 8118 imgs | 0.164 con grieta
{'Walls': 3276, 'Pavements': 2574, 'Decks': 2268}
test 32 fotos | 7795 imgs | 0.177 con grieta
{'Pavements': 3510, 'Walls': 3025, 'Decks': 1260}


#Modelo 1: CNN Propia


In [ ]:
def build_custom_cnn(input_shape=(128, 128, 3)):
    # CNN propia sin preentrenar
    model = models.Sequential([
        # Bloque 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(), # Regularización BatchNorm
        layers.MaxPooling2D((2, 2)),

        # Bloque 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Bloque 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Global Average Pooling (GAP) reemplaza al Flatten tradicional
        layers.GlobalAveragePooling2D(),

        # Regularización solicitada
        layers.Dropout(0.4),

        # Capa de salida binaria (1 neurona, sigmoide para clasificar grieta/no grieta)
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_custom_cnn()

#ResNET 50

In [ ]:
#@title Modelo base
def build_resnet50_base(input_shape=(128, 128, 3)):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir el nuevo modelo agregando la cabeza personalizada
    modelResNET = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), # Requisito de regularización base para los experimentos
        layers.Dense(1, activation='sigmoid') # Salida binaria (grieta / no grieta)
    ])

    return model, base_model

# Crear la arquitectura inicial
model, base_model = build_resnet50_base()
model.summary()

In [ ]:
#@title Modelo completamente congelado

# 1. Congelar toda la ResNet50
base_model.trainable = False

# 2. Compilar (LR = 1e-3)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Ejecutar model.fit(..., class_weight=class_weight_dict)

In [ ]:
#@title Ultima capa descongelada


# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Iterar y congelar todo excepto el último bloque (sin tocar los BN)
for layer in base_model.layers:
    # Si la capa pertenece al bloque 5 y NO es BatchNormalization, se descongela
    if layer.name.startswith('conv5_block') and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        # Congelamos el resto de bloques y TODAS las capas BatchNormalization
        layer.trainable = False

# 3. Compilar (LR = 1e-4)
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
#@title Modelo descongelado a excepción de BN


# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Congelar EXCLUSIVAMENTE las capas BatchNormalization
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilar (LR = 1e-5 y adición de weight decay)
# Nota: En Keras modernos, el optimizador Adam acepta el parámetro weight_decay directamente
model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

#EfficientNetB0

In [ ]:
def build_efficientnet_base(input_shape=(128, 128, 3)):
    # 1. Instanciar EfficientNetB0 preentrenada en ImageNet
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir la cabeza personalizada
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), # Regularización requerida
        layers.Dense(1, activation='sigmoid') # Clasificación binaria
    ])

    return model, base_model

model, base_model = build_efficientnet_base()

In [ ]:
#@title Modelo completamente congelado


# 1. Congelar toda la base
base_model.trainable = False

# 2. Compilar con LR = 1e-3
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Ejecutas model.fit(..., class_weight=class_weight_dict)

In [ ]:
#@title Descongelamiento gradual

# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Descongelar solo el último bloque (block7) y dejar todo lo demás congelado
for layer in base_model.layers:
    if layer.name.startswith('block7') and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Compilar con LR = 1e-4
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
# Ejecutas model.fit(...) unas cuantas épocas

In [ ]:
#@title Completamente descongelado
# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Congelar EXCLUSIVAMENTE las capas BatchNormalization
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilar con LR = 1e-5 y agregar weight_decay
model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
# Ejecutas model.fit(...)

#VGG16

In [ ]:
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
model_vgg = models.Sequential([
    base_vgg,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5), # Especificado en la tabla
    layers.Dense(1, activation='sigmoid')
])

# 2. Descongelar solo los últimos 2 bloques (block4 y block5)
base_vgg.trainable = True
for layer in base_vgg.layers:
    if layer.name.startswith('block4') or layer.name.startswith('block5'):
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Compilación (LR = 1e-4)
model_vgg.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

#DenseNet121

In [ ]:
# 1. Base y arquitectura
base_dense = DenseNet121(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
model_dense = models.Sequential([
    base_dense,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4), # Especificado en la tabla
    layers.Dense(1, activation='sigmoid')
])

# 2. Descongelar último bloque (conv5) protegiendo el BN
base_dense.trainable = True
for layer in base_dense.layers:
    if layer.name.startswith('conv5') and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Compilación (LR = 1e-4)
model_dense.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

#ConvNeXt

In [ ]:
# 1. Arquitectura con Data Augmentation integrado
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])

base_convnext = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
model_convnext = models.Sequential([
    data_augmentation,
    base_convnext,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4), # Dropout general
    layers.Dense(1, activation='sigmoid')
])

# 2. Fine-tuning completo protegiendo LayerNormalization
base_convnext.trainable = True
for layer in base_convnext.layers:
    if isinstance(layer, tf.keras.layers.LayerNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilación (LR = 1e-5 y Weight Decay)
model_convnext.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

MobileNetV3

In [ ]:
def build_mobilenetv3_base(input_shape=(128, 128, 3)):
    # 1. Instanciar MobileNetV3 (Large) preentrenada en ImageNet
    base_model = MobileNetV3Large(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir la cabeza personalizada
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3), # Regularización requerida en la tabla
        layers.Dense(1, activation='sigmoid') # Clasificación binaria
    ])

    return model, base_model

model, base_model = build_mobilenetv3_base()

In [ ]:
# 1. Congelar toda la base
base_model.trainable = False

# 2. Compilar (LR = 1e-3)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Descongelar últimos bloques (ej. expanded_conv_12 al 14 y Conv_1/Conv_2 finales)
for layer in base_model.layers:
    # Identificar las capas finales de MobileNetV3Large
    if ('expanded_conv_12' in layer.name or
        'expanded_conv_13' in layer.name or
        'expanded_conv_14' in layer.name or
        'Conv_1' in layer.name or
        'Conv_2' in layer.name):

        # Congelar BatchNormalization dentro de estos bloques
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True
    else:
        layer.trainable = False # Congelar el resto de la red temprana

# 3. Compilar (LR = 1e-4)[cite: 16]
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Congelar EXCLUSIVAMENTE las capas BatchNormalization
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilar con LR = 1e-5 y agregar weight_decay
model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)